# 계절별 산불 발생 현황 분석 (1997 ~ 2025)
동국대 AI캠퍼스 · matplotlib 프로젝트 예제

- 데이터: KOSIS 계절별 산불 발생 현황 (금년 · 10년평균)
- 흐름: 문제정의 → 데이터 준비(제공) → 계절별 추세 → 금년 vs 평년 → 결론

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------
# 한글 폰트 설정 (OS 자동 감지)
# ---------------------------------------
import platform
_os = platform.system()
if _os == 'Windows':
    plt.rc('font', family='Malgun Gothic')     # 윈도우
elif _os == 'Darwin':
    plt.rc('font', family='AppleGothic')        # 맥
else:
    plt.rc('font', family='NanumGothic')        # 리눅스 / 구글 코랩
plt.rc('axes', unicode_minus=False)
plt.rc('figure', figsize=(12, 5))

## 문제 정의
- 산불은 **어느 계절**에 가장 많이 발생하나?
- 계절별 비중은 해마다 어떻게 달라지나?
- **2025년**은 **10년 평년**과 비교해 어땠나?

## 데이터 준비 (제공)
> **데이터 준비 코드는 미리 작성해 두었습니다(다단 헤더 처리는 pandas 심화 내용).**
>  아래 표(`cur`(금년)·`avg`(평년))는 **연도별로 정리된 쉬운 컬럼**이니, 우리는 **시각화에 집중**합니다.

In [ ]:
# ----------------------------------------
# [제공] 로드 + 정제 + 컬럼 정리
# ----------------------------------------
data_file = 'data/산불데이터/계절별_산불발생_현황_20260712203157.csv'
raw = pd.read_csv(data_file, header=[0, 1])
raw = raw.set_index(('시점', '시점'))
raw.index.name = '연도'
raw = raw.apply(lambda s: pd.to_numeric(
    s.astype(str).str.replace('-', '0').str.replace(',', ''), errors='coerce')).fillna(0)

# 금년(cur) / 10년평균(avg) 블록을 쉬운 컬럼명으로 정리
cols = ['합계', '봄', '여름', '가을', '겨울']
cur = raw['금년'].copy();    cur.columns = cols
avg = raw['10년평균'].copy(); avg.columns = cols
cur.head()

## 계절별 발생 추세 (금년 기준)

In [ ]:
seasons = ['봄', '여름', '가을', '겨울']
season_colors = ['#F28CB1', '#4EA8DE', '#D2691E', '#6C757D']
season_mean = cur[seasons].mean()
season_mean.round(1)

#### 계절별 평균 비중 (파이)

In [ ]:
plt.figure(figsize=(6, 6))
plt.pie(season_mean, labels=seasons, autopct='%.1f%%',
        colors=season_colors, startangle=90, counterclock=False)
plt.title('계절별 산불 발생 평균 비중 (1997~2025)')
plt.show()

#### 연도별 계절 구성 (누적 막대)

In [ ]:
cur[seasons].plot(kind='bar', stacked=True, figsize=(14, 5), color=season_colors)
plt.title('연도별 계절 구성 (발생 건수)')
plt.xlabel('연도'); plt.ylabel('건수'); plt.xticks(rotation=45)
plt.legend(title='계절', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2025년 : 금년 vs 10년 평년

In [ ]:
cur_2025 = cur.loc[2025, seasons]
avg_2025 = avg.loc[2025, seasons]

x = np.arange(len(seasons))
plt.figure(figsize=(9, 5))
plt.bar(x - 0.2, cur_2025.values, width=0.4, label='2025 금년', color='#E67E22')
plt.bar(x + 0.2, avg_2025.values, width=0.4, label='10년 평균', color='#95A5A6')
plt.xticks(x, seasons)
plt.title('2025년 계절별 발생 : 금년 vs 10년 평균')
plt.ylabel('건수'); plt.legend()
plt.show()

In [ ]:
print('2025 금년 총 발생:', int(cur.loc[2025, '합계']))
print('10년 평균 총 발생:', int(avg.loc[2025, '합계']))

## 결론
- 산불은 건조한 **봄(3~5월)**에 압도적으로 많이 발생한다. 겨울이 그 다음.
- **2025년은 발생 건수 자체는 평년보다 적었다**(459건 < 평균 529건). 그럼에도 피해는 역대 최대(→ 산불1) → **소수의 대형 산불**이 피해를 좌우.
- '몇 건 났나'보다 '한 건이 얼마나 커지나'가 피해를 결정한다.